[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saranabhani/rug-analysing-data/blob/main/week5/k_means_svm.ipynb)

In [ ]:
import pandas as pd
import spacy
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from sklearn.cluster import KMeans
from wordcloud import WordCloud

In [ ]:
nlp = spacy.load('en_core_web_sm')

# Load and explore the data

In [ ]:
!git clone https://github.com/saranabhani/rug-analysing-data.git
%cd rug-analysing-data/week5

In [ ]:
data_file_path = 'news_data.csv'
data = pd.read_csv(data_file_path)

In [ ]:
data

In [ ]:
data= data.dropna(subset=['headline', 'short_description', 'category'])

In [ ]:
data['category'].value_counts()

# Preprocess the text

In [ ]:
def preprocess_text(text):
    # lowercase and tokenize
    doc = nlp(text)
    # Lemmatize and remove stopwords, punctuations, and numbers, and return the tokens
    return ' '.join([token.lemma_.lower() for token in doc if token.is_alpha and not token.is_stop])

In [ ]:
headlines = data['headline']
# Extract short descriptions
short_descriptions = data['short_description']
# Extract categories
categories = data['category']

In [ ]:
headlines_processed = [preprocess_text(headline) for headline in headlines]
short_descriptions_processed = [preprocess_text(short_description) for short_description in short_descriptions]
headline_desc_processed = [headline + ' ' + short_desc for headline, short_desc in zip(headlines_processed, short_descriptions_processed)]

# TF-IDF Vectorization

In [ ]:
vectorizer = TfidfVectorizer(min_df = 5, max_df = 0.8, max_features= 3000)
X = vectorizer.fit_transform(headline_desc_processed)

# Train-test split and SVM classification

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(X, categories, test_size=0.2, random_state=42, stratify=categories)

In [ ]:
svc = SVC(kernel='linear', random_state=42)
svc.fit(x_train.toarray(), y_train)

In [ ]:
y_pred = svc.predict(x_test.toarray())

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=svc.classes_)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=svc.classes_, yticklabels=svc.classes_)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.show()

# K-means clustering on the TF-IDF vectors to find common themes in the news articles

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42)
kmeans.fit(X)

In [ ]:
def plot_top_words(vectorizer, kmeans, n_top_words=10):
    feature_names = vectorizer.get_feature_names_out()
    for i in range(kmeans.n_clusters):
        cluster_center = kmeans.cluster_centers_[i]
        top_indices = np.argsort(cluster_center)[::-1][:n_top_words]
        top_words = [feature_names[index] for index in top_indices]
        plt.figure(figsize=(8, 4))
        plt.barh(top_words, cluster_center[top_indices])
        plt.title(f"Top words in Cluster {i}")
        plt.xlabel("Score")
        plt.gca().invert_yaxis()
        plt.show()

In [ ]:
plot_top_words(vectorizer, kmeans, n_top_words=15)

In [ ]:
def plot_word_cloud(vectorizer, kmeans, n_top_words=100):
    feature_names = vectorizer.get_feature_names_out()
    for i in range(kmeans.n_clusters):
        cluster_center = kmeans.cluster_centers_[i]
        top_indices = np.argsort(cluster_center)[::-1][:n_top_words]
        top_words = {feature_names[index]: cluster_center[index] for index in top_indices}
        wordcloud = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(top_words)
        plt.figure(figsize=(10, 5))
        plt.imshow(wordcloud, interpolation='bilinear')
        plt.axis('off')
        plt.title(f"Word Cloud for Cluster {i}")
        plt.show()

In [ ]:
plot_word_cloud(vectorizer, kmeans, n_top_words=100)